In [20]:
# conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia
# pip install transformers datasets faiss-cpu 
# pip install sentence-transformers 
# pip install tqdm 
# pip install scikit-learn 
# pip install flask 
# pip install jupyter notebook
# pip install python-dotenv

# pip install langchain_core langchain_groq
# pip install Pillow pytesseract python-docx pandas python-pptx
# pip install pytesseract pillow langchain
# pip install -U langchain-community
# pip install pythainlp
# pip install ollama
# pip install chromadb
# pip install -U langchain-ollama
# pip install pypdf
# pip install openpyxl
# pip install pdfplumber camelot-py 
# pip install chromadb==0.4.24
# pip install unstructured pdfminer.six python-magic-bin pytesseract pillow beautifulsoup4 langchain pythainlp requests
# %pip install "unstructured[all-docs]"
# conda install -c conda-forge poppler -y

In [21]:
import pandas as pd
import numpy as np

In [22]:
from dotenv import load_dotenv
import os

load_dotenv() 
# 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
# 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
# langchain_token = os.getenv("LC_TOKEN")
# groq_token = os.getenv("GROQ_TOKEN")

langchain_token = "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"
groq_token = "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"

print("LC_TOKEN:", langchain_token)
print("GROQ_TOKEN:", groq_token)


LC_TOKEN: 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
GROQ_TOKEN: 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


In [23]:
import os

#Langsmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = langchain_token

In [24]:


# Then import and set up Groq
import os
from langchain_groq import ChatGroq

# Set your Groq API key
os.environ["GROQ_API_KEY"] = groq_token

# Initialize the LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

In [25]:
import os
print(os.getcwd())

d:\Project\is-rag-agro\notebooks


In [26]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU Available: True
GPU Name: NVIDIA GeForce RTX 3080


In [27]:
import sys
print(sys.executable)
import importlib.metadata
print(importlib.metadata.version("unstructured"))

c:\Users\Biabya\anaconda3\envs\rag-agro\python.exe
0.18.15


In [30]:
# --- SETTINGS ---
import os, io, requests, re
from typing import List
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# LangChain
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Unstructured partitioners (เลือกใช้ตามชนิดไฟล์)
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx
from unstructured.partition.pptx import partition_pptx
from unstructured.partition.html import partition_html
from unstructured.partition.xlsx import partition_xlsx
from unstructured.partition.csv import partition_csv
from unstructured.partition.text import partition_text
from unstructured.partition.image import partition_image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from math import ceil
from pypdf import PdfReader

# OCR (ใช้ tesseract ภาษาไทย+อังกฤษ)
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"  # แก้ path ตามเครื่อง

# ---------- Helper: ทำความสะอาดข้อความ OCR ภาษาไทย ----------
import pythainlp
import unicodedata
from pythainlp.tokenize import word_tokenize
def clean_thai_ocr(text: str) -> str:
    # นอร์มยูนิโค้ด
    text = unicodedata.normalize("NFC", text)

    # รวมตัวอักษรไทยที่โดนแยกช่องว่างระหว่างพยางค์/สระ/วรรณยุกต์
    # เช่น ช า ระ → ชำระ, ท า การ → ทำการ
    text = re.sub(r'([ก-๙เแโใไ]{1})\s+([่-๋ัิุูำ])', r'\1\2', text)     # ตัว+วรรณยุกต์
    text = re.sub(r'([เแโใไ])\s+([ก-๙])', r'\1\2', text)                 # สระหน้า + ตัว
    text = re.sub(r'([ก-๙])\s+([ก-๙])', r'\1\2', text)                   # ตัวติดตัว (แกนหลัก)

    # แมพคำที่พบบ่อยจาก OCR
    fixes = {
        # r'ช\s*า\s*ระ': 'ชำระ',
        # r'ท\s*า\s*การ': 'ทำการ',
        # r'ล\s*ง\s*ท\s*ะ\s*เ\s*บี\s*ย\s*น': 'ลงทะเบียน',
        # r'ว\s*ัน\s*ท\s*ำ\s*ก\s*า\s*ร': 'วันทำการ',
    }
    for patt, repl in fixes.items():
        text = re.sub(patt, repl, flags=re.IGNORECASE)

    # เกลี่ยช่องว่างทั่วไป
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_and_tokenize_thai(text: str) -> str:
    return ' '.join(word_tokenize(clean_thai_ocr(text), keep_whitespace=False))

# ---------- Helper: แปลง Unstructured Elements -> LangChain Documents ----------
def elements_to_docs(elements, source: str) -> List[Document]:
    docs = []
    for el in elements:
        text = (getattr(el, "text", "") or "").strip()
        if not text:
            if getattr(el, "category", "") in ("Table", "Image"):
                text = f"[{el.category}]"
            else:
                continue

        md = getattr(el, "metadata", None)
        meta = {
            "source": source,
            "category": getattr(el, "category", None),
            # Unstructured ส่วนใหญ่เก็บไว้ใน el.metadata.*
            "filename": getattr(md, "filename", None) if md else None,
            "page_number": getattr(md, "page_number", None) if md else None,
            "section": getattr(md, "section", None) if md else None,
            # HTML มักจะมี .links แทนที่จะเป็น link_urls
            "links": getattr(md, "links", None) if md else None,
            # บาง element อาจมี text_as_html
            "text_as_html": getattr(el, "text_as_html", None),
        }
        docs.append(Document(page_content=text, metadata={k: v for k, v in meta.items() if v}))
    return docs

# ---------- Loaders ด้วย Unstructured ----------
def _partition_pdf_range(file_path: str, p: int):
    # พยายามอ่านหน้าเดียวแบบข้อความก่อน
    try:
        return partition_pdf(
            filename=file_path,
            include_metadata=True,
            infer_table_structure=True,
            strategy="hi_res",
            languages=["tha", "eng"],
            start_page_number=p,
            end_page_number=p,
        )
    except Exception:
        # ถ้าพัง หน้าเดียว OCR-only
        return partition_pdf(
            filename=file_path,
            include_metadata=True,
            infer_table_structure=False,
            strategy="ocr_only",
            languages=["tha", "eng"],
            start_page_number=p,
            end_page_number=p,
        )

def _sanitize_pdf(in_path: str) -> str:
    # ซ่อม pdf (repack) ช่วยเคลียร์ content stream เพี้ยนๆ
    try:
        import pikepdf, os, tempfile
        out_path = os.path.join(tempfile.gettempdir(), f"sanitized_{os.path.basename(in_path)}")
        with pikepdf.open(in_path) as pdf:
            pdf.save(out_path, linearize=True)
        return out_path
    except Exception:
        return in_path  # ถ้าซ่อมไม่ได้ ใช้ไฟล์เดิม



# --- ตัวช่วยเล็ก ๆ ---
def _quick_text_len(reader: PdfReader, p_idx: int) -> int:
    try:
        t = reader.pages[p_idx].extract_text() or ""
        return len(t.strip())
    except Exception:
        return 0

def _partition_pdf_batch(
    file_path: str,
    start_p: int,
    end_p: int,
    use_hi_res: bool = False,
    do_ocr_only: bool = False,
    infer_table: bool = False,
):
    # เลือกระหว่าง fast / hi_res / ocr_only ตามธง
    if do_ocr_only:
        strategy = "ocr_only"
        infer_table_structure = False
    elif use_hi_res:
        strategy = "hi_res"
        infer_table_structure = infer_table
    else:
        strategy = "fast"
        infer_table_structure = infer_table

    return partition_pdf(
        filename=file_path,
        include_metadata=True,
        strategy=strategy,
        languages=["tha", "eng"],
        infer_table_structure=infer_table_structure,
        start_page_number=start_p,
        end_page_number=end_p,
        # เคล็ดลับความเร็ว: ไม่ดึงรูปออกมา
        extract_images_in_pdf=False,
        # ไม่ต้องคั่นหน้า
        include_page_breaks=False,
    )

def load_pdf_unstruct(path: str) -> List[Document]:
    docs: List[Document] = []

    # รอบ 1: ดึงแบบเร็ว ไม่ OCR ก่อน
    elements = partition_pdf(
        filename=path,
        strategy="fast",
        include_metadata=True,
        infer_table_structure=False,   # เปิดเฉพาะตอนจำเป็น (ช้า)
        extract_images_in_pdf=False,
        languages=["tha", "eng"],
        include_page_breaks=False,
    )

    total_chars = sum(len(getattr(e, "text", "") or "") for e in elements)

    # ถ้าข้อความน้อยมาก → ลอง OCR ทั้งเล่ม
    if total_chars < 500:
        elements = partition_pdf(
            filename=path,
            strategy="ocr_only",
            include_metadata=True,
            infer_table_structure=True,    # OCR + table (ยอมช้าขึ้นเฉพาะกรณีนี้)
            extract_images_in_pdf=False,
            languages=["tha", "eng"],
            include_page_breaks=False,
        )

    for e in elements:
        text = (getattr(e, "text", "") or "").strip()
        if not text:
            continue
        md = getattr(e, "metadata", None)
        page = getattr(md, "page_number", None) if md else None
        etype = getattr(e, "category", None) or e.__class__.__name__
        docs.append(Document(
            page_content=text,
            metadata={
                "source": path,
                "page_number": page,    # ใช้คีย์มาตรฐาน
                "category": etype
            }
        ))
    return docs



def load_docx_unstruct(file_path: str) -> List[Document]:
    elements = partition_docx(filename=file_path, include_metadata=True)
    return elements_to_docs(elements, source=file_path)

def load_pptx_unstruct(file_path: str) -> List[Document]:
    elements = partition_pptx(filename=file_path, include_metadata=True)
    return elements_to_docs(elements, source=file_path)

def load_xlsx_unstruct(file_path: str) -> List[Document]:
    # partition_xlsx จะดึงเป็น elements (cell/table) + metadata
    elements = partition_xlsx(filename=file_path, include_metadata=True)
    return elements_to_docs(elements, source=file_path)

def load_csv_unstruct(file_path: str) -> List[Document]:
    elements = partition_csv(filename=file_path, include_metadata=True)
    return elements_to_docs(elements, source=file_path)

def load_txt_unstruct(file_path: str) -> List[Document]:
    elements = partition_text(filename=file_path, include_metadata=True, encoding="utf-8")
    return elements_to_docs(elements, source=file_path)

def load_img_unstruct(file_path: str) -> List[Document]:
    # ✅ ใช้ partition_image ถูกตัวแล้ว และใช้ languages (ไม่ใช่ ocr_languages)
    elements = partition_image(
        filename=file_path,
        include_metadata=True,
        strategy="ocr_only",
        languages=["tha", "eng"],
    )
    docs = elements_to_docs(elements, source=file_path)
    # ทำความสะอาด/ตัดคำไทย (ถ้าต้องการ)
    for d in docs:
        d.page_content = clean_and_tokenize_thai(d.page_content)
    return docs

# ---------- Web Ingestion ด้วย Unstructured (HTML + OCR รูปในเพจ) ----------
def _normalize_img_src(tag):
    # รองรับ lazy-load ที่ใช้ data-src / data-original / srcset
    for key in ("data-src", "data-original", "data-lazy-src"):
        if tag.get(key):
            tag["src"] = tag.get(key)
            break
    # ดึงจาก srcset ตัวแรกถ้าจำเป็น
    if not tag.get("src") and tag.get("srcset"):
        tag["src"] = tag["srcset"].split(",")[0].split()[0]

def load_web_to_documents(url: str, content_selector: str = "body") -> List[Document]:
    try:
        r = requests.get(url, timeout=15, headers={
            "User-Agent": "Mozilla/5.0 (compatible; ingest-bot/1.0)"
        })
        r.raise_for_status()
        r.encoding = r.apparent_encoding or r.encoding  # กันกรณี charset เพี้ยน
    except Exception as e:
        print(f"[WEB] load fail: {url} -> {e}")
        return []

    soup = BeautifulSoup(r.text, "html.parser")

    # ลบ script/style/comment เพื่อลด noise
    for s in soup(["script", "style", "noscript"]):
        s.decompose()

    section = soup.select_one(content_selector) or soup

    # แก้ lazy-load รูป
    for imgtag in section.find_all("img"):
        _normalize_img_src(imgtag)

    html_str = str(section)

    # ส่งเฉพาะส่วนที่สนใจให้ unstructured
    elements = partition_html(text=html_str, include_metadata=True)
    docs = elements_to_docs(elements, source=url)

    # OCR รูปใน section
    imgs = section.find_all("img")
    for imgtag in imgs:
        src = imgtag.get("src")
        if not src:
            continue
        full = urljoin(url, src)
        # ข้าม data:image เพื่อประหยัดเวลา
        if full.startswith("data:"):
            continue
        try:
            ir = requests.get(full, timeout=10, headers={"User-Agent": "Mozilla/5.0 (compatible; ingest-bot/1.0)"})
            ir.raise_for_status()

            # กันไฟล์ใหญ่เกิน (เช่น > 8MB)
            if int(ir.headers.get("Content-Length", 0)) > 8_000_000:
                docs.append(Document(page_content=f"[OCR skip: {full} - image too large]", metadata={"source": url}))
                continue

            elements_img = partition_image(
                file=io.BytesIO(ir.content),
                include_metadata=True,
                languages=["tha", "eng"],
                strategy="ocr_only",
            )
            ocr_docs = elements_to_docs(elements_img, source=f"{url}#img:{full}")
            # ทำความสะอาด/ตัดคำไทย ทั้งข้อความจากรูป
            for d in ocr_docs:
                d.page_content = clean_and_tokenize_thai(d.page_content)
            docs.extend(ocr_docs)
        except Exception as e:
            docs.append(Document(page_content=f"[OCR fail: {full} - {e}]", metadata={"source": url}))
    return docs

def load_pdf_ocr_tables(path: str) -> List[Document]:
    """
    OCR ทั้งเล่ม (ไทย+อังกฤษ) + เดาโครงสร้างตาราง
    - เก็บตารางเป็น HTML (text_as_html) เพื่อความแม่นของคอลัมน์/เซลล์
    - อื่น ๆ เก็บเป็นข้อความปกติ
    """
    elements = partition_pdf(
        filename=path,
        strategy="ocr_only",
        include_metadata=True,
        infer_table_structure=True,   # <<< สำคัญเพื่อให้ตารางถูกแยกเป็นโครงสร้าง
        languages=["tha", "eng"],
        extract_images_in_pdf=False,
        include_page_breaks=False,
    )

    docs: List[Document] = []
    for e in elements:
        md = getattr(e, "metadata", None)
        page = getattr(md, "page_number", None) if md else None
        cat  = getattr(e, "category", None) or e.__class__.__name__

        # ใช้ HTML สำหรับตาราง (แม่นกว่า text ธรรมดา)
        html = getattr(e, "text_as_html", None)
        text = (getattr(e, "text", "") or "").strip()
        content = html if (cat == "Table" and html) else text
        if not content:
            continue

        docs.append(Document(
            page_content=content,
            metadata={
                "source": path,
                "page_number": page,
                "category": cat
            }
        ))
    return docs

# ---------- รวม Loader หลายโฟลเดอร์ ----------
import time

def load_documents_from_folders(folder_paths: List[str]) -> List[Document]:
    docs: List[Document] = []
    for folder in folder_paths:
        start = time.perf_counter()
        count_before = len(docs)

        print(f"--- Start ingest folder: {folder} ---")
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            low = filename.lower()
            try:
                if low.endswith(".pdf"):
                    docs.extend(load_pdf_ocr_tables(file_path))
                elif low.endswith(".docx"):
                    docs.extend(load_docx_unstruct(file_path))
                elif low.endswith(".pptx"):
                    docs.extend(load_pptx_unstruct(file_path))
                elif low.endswith(".xlsx"):
                    docs.extend(load_xlsx_unstruct(file_path))
                elif low.endswith(".csv"):
                    docs.extend(load_csv_unstruct(file_path))
                elif low.endswith((".jpg", ".jpeg", ".png", ".webp", ".tiff")):
                    docs.extend(load_img_unstruct(file_path))
                elif low.endswith((".txt", ".md")):
                    docs.extend(load_txt_unstruct(file_path))
                else:
                    continue
            except Exception as e:
                docs.append(Document(
                    page_content=f"[INGEST FAIL] {file_path}: {e}",
                    metadata={"source": file_path}
                ))

        elapsed = time.perf_counter() - start
        count_added = len(docs) - count_before
        print(f"--- Done folder: {folder} | {count_added} docs | {elapsed:.2f} sec ---")

    return docs


In [32]:
# ---------- ใช้งาน ----------
# folder_paths = ['./Train/docx', './Train/jpg', './Train/pdf', './Train/pptx', './Train/xlsx']
# folder_paths = ['./Train/docx', './Train/jpg', './Train/pdf', './Train/pptx', './Train/xlsx']
folder_paths = ['./Train/testing']  # แก้ตามจริง

# Ensure all folders exist
for folder in folder_paths:
    if not os.path.exists(folder):
        os.makedirs(folder, exist_ok=True)

docs = load_documents_from_folders(folder_paths)
for d in docs:
    d.page_content = clean_thai_ocr(d.page_content)

# ถ้าจะดึงเว็บเพิ่ม
web_urls = [
    # "https://registrar.ku.ac.th/calendar",
    # "https://agro.ku.ac.th/th/academics",
]
for url in web_urls:
    docs.extend(load_web_to_documents(url, content_selector='body'))

table_docs, text_docs = [], []
for d in docs:
    if "<table" in d.page_content.lower() and d.metadata.get("category") == "Table":
        table_docs.append(d)     # เก็บทั้งโต๊ะเป็นหนึ่งชิ้น
    else:
        text_docs.append(d)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n","\n","。","！","？",".","!","?"," ",""],
    keep_separator=False
)
splits = table_docs + splitter.split_documents(text_docs)


print("Total docs:", len(docs), "Splits:", len(splits))
print("Docs (raw):", len(docs))
print("Splits:", len(splits))
print("Sample:", splits[0].metadata, (splits[0].page_content[:300] + "…"))

--- Start ingest folder: ./Train/testing ---
--- Done folder: ./Train/testing | 492 docs | 1.48 sec ---
Total docs: 492 Splits: 498
Docs (raw): 492
Splits: 498
Sample: {'source': './Train/testing\\หลักสูตร อก มก no formatting.docx', 'category': 'UncategorizedText', 'filename': 'หลักสูตร อก มก no formatting.docx'} หลักสูตรที่เปิดสอน…


In [33]:
print("start")

# --- imports ที่จำเป็น ---
import shutil
import gradio as gr
import chromadb
import ollama  # ต้องมี `pip install ollama` และมีโมเดลในเครื่อง
from chromadb.config import Settings
from langchain_community.vectorstores import Chroma
# ใช้ตัวใหม่เพื่อลด Deprecation Warning:
# pip install -U langchain-ollama
try:
    from langchain_ollama import OllamaEmbeddings
except ImportError:
    # ถ้ายังไม่ได้ลง langchain-ollama ให้ fallback ใช้ตัวเดิมชั่วคราว
    from langchain_community.embeddings import OllamaEmbeddings

# --- เคลียร์/ตั้งค่าโฟลเดอร์ Chroma ---
shutil.rmtree("./chroma_db_nomic", ignore_errors=True)

# --- สร้าง embeddings ---
emb = OllamaEmbeddings(model="nomic-embed-text")  # หรือ "mxbai-embed-large"

client_settings = Settings(
    is_persistent=True,
    persist_directory="./chroma_db_nomic",
    anonymized_telemetry=False,
    chroma_db_impl="duckdb+parquet",  # <— สำคัญ
)

# NOTE: ต้องมีตัวแปร `splits` ที่คุณสร้างจาก TextSplitter มาก่อนหน้านี้

# --- New Chroma client ---
chroma_client = chromadb.PersistentClient(
    path="./chroma_db_nomic"
)

# --- LangChain wrapper ---
vectorstore = Chroma(
    client=chroma_client,
    collection_name="agro_nomic_v1",
    embedding_function=emb,
)
vectorstore.add_documents(splits)



print("embedding ready:", emb)

# --- LLM ผ่าน Ollama ---
def ollama_llm(question, context):
    # ตรวจชื่อโมเดลให้ตรงกับที่คุณมีในเครื่อง เช่น "llama3:8b", "gemma2:9b", "qwen2.5:7b"
    model_name = "gemma3:27b" 
    guardrail = (
        # "คุณเป็นผู้ช่วยตอบคำถามจากฐานข้อมูลเท่านั้น "
        # "ให้ใช้ข้อมูลที่อยู่ใน 'เนื้อหา (Context)' ด้านล่างเท่านั้นในการตอบ "
        # "ห้ามเติมความรู้ภายนอกหรือคาดเดาเอง "
        # "ถ้าไม่พบคำตอบใน Context ให้ตอบว่า: \"ไม่พบข้อมูลในฐานข้อมูลสำหรับคำถามนี้\".\n\n"
        # "รูปแบบคำตอบ: ตอบสั้น กระชับ เป็นข้อ ๆ (ถ้าเหมาะสม) และแนบอ้างอิงท้ายบรรทัดเช่น [1],[2]\n"
    )
    formatted_prompt = f"""{guardrail}
คำถาม: {question}

เนื้อหา (Context):
{context}


โปรดตอบเป็นภาษาไทย สรุปข้อเท็จจริงสำคัญ กระชับ ชัดเจน
"""
    resp = ollama.chat(model=model_name, messages=[{"role": "user", "content": formatted_prompt}])
    print("Formatted Prompt:", formatted_prompt)
    print("Count Tokens:", len(formatted_prompt.split()))
    # print('token count:', resp['usage']['total_tokens'])
    return resp["message"]["content"]

# --- Retriever ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def rag_chain(question):
    # รองรับทั้ง API เก่า/ใหม่ของ LangChain
    try:
        docs = retriever.invoke(question)  # LangChain >= 0.2
    except Exception:
        docs = retriever.get_relevant_documents(question)  # เวอร์ชันเก่า
    context = "\n\n".join(d.page_content for d in docs)
    # print ออกมา
    print(f"🔎 Query: {question}")
    print(f"📌 Retrieved {len(docs)} documents:\n")

    for i, d in enumerate(docs, 1):
        print(f"--- Document {i} ---")
        print("Meta:", d.metadata)
        print("Content:", d.page_content[:300], "...\n")  # ตัดข้อความสั้น ๆ
    return ollama_llm(question, context)

def get_important_facts(question):
    return rag_chain(question)

# --- Gradio app ---
iface = gr.Interface(
    fn=get_important_facts,
    inputs=gr.Textbox(lines=2, placeholder="พิมพ์คำถามของคุณที่นี่ (ไทย/อังกฤษได้)"),
    outputs="text",
    title="ถามข้อมูลเกี่ยวกับ Agro (RAG)",
    description="ถามคำถามเกี่ยวกับเนื้อหา แล้วรับคำตอบสรุปเป็นข้อเท็จจริงสำคัญ (ตอบเป็นไทย)",
)

iface.launch()  # ใน Jupyter ถ้าอยากเห็นลิงก์ภายนอกเพิ่ม share=True
print("end")


start


C:\Users\Biabya\AppData\Local\Temp\ipykernel_10928\3922101397.py:39: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


embedding ready: model='nomic-embed-text' validate_model_on_init=False base_url=None client_kwargs={} async_client_kwargs={} sync_client_kwargs={} mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None keep_alive=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


end


🔎 Query: หลักสูตรที่เปิดสอนระดับปริญญาตรีมีอะไรบ้าง
📌 Retrieved 5 documents:

--- Document 1 ---
Meta: {'source': './Train/testing\\หลักสูตร อก มก no formatting.docx', 'category': 'UncategorizedText', 'filename': 'หลักสูตร อก มก no formatting.docx'}
Content: เทคโนโลยีการบรรจุ ...

--- Document 2 ---
Meta: {'filename': 'หลักสูตร อก มก no formatting.docx', 'source': './Train/testing\\หลักสูตร อก มก no formatting.docx', 'category': 'UncategorizedText'}
Content: ะตัวเร่งทางชีวภาพสำหรับกระบวนการผลิตเข้าใจถึงกรรมวิธีการผลิตที่เหมาะสมต่อการทำงานของจุลินทรีย์และตัวเร่งทางชีวภาพตลอดจนการออกแบบและควบคุมเครื่องมือซึ่งใช้ในกระบวนการผลิตภาควิชาฯจัดการเรียนการสอนระดับปริญญาตรีและบัณฑิตศึกษาในหลีกสูตรปกติและหลักสูตรนานาชาติทั้งระดับปริญญาโทและปริญญาเอกเพื่อสนองตอบต่อก ...

--- Document 3 ---
Meta: {'filename': 'หลักสูตร อก มก no formatting.docx', 'category': 'UncategorizedText', 'source': './Train/testing\\หลักสูตร อก มก no formatting.docx'}
Content: เป้าหมายของภาควิชาคือการสร้างบุคลากรหลังปริญญาตรีท